In [1]:
# This notebook shows the implementation of a trigram model (part of makemore's exercise)

In [2]:
words = open("names.txt", "r").read().splitlines()

In [3]:
min(len(word) for word in words)

2

In [4]:
max(len(word) for word in words)

15

In [5]:
# getting all the charecters that are unique in the names dataset
chars = list(sorted(set("".join(words))))
# each characters associated with an index to later create a probability matrix out of it.
stoi = {s:i+1 for i,s in enumerate(chars)}
# add . as it's a special token used to denote start and end of word.
stoi['.'] = 0
# reverse stoi list to get i that maps to its respective s
itos = {i:s for s,i in stoi.items()}

In [7]:
len(chars)

26

In [8]:
import torch

In [9]:
pmatrx = torch.zeros((27, 27, 27), dtype=torch.int32)

In [10]:
# padding with tokens to indicate start and end words.
for w in words:
    char = ['.'] + ['.'] + list(w) + ['.']
    for c1, c2, c3 in zip(char, char[1:], char[2:]):
        ix1 = stoi[c1]
        ix2 = stoi[c2]
        ix3 = stoi[c3]
        pmatrx[ix1, ix2, ix3] += 1

In [11]:
import matplotlib.pyplot as plt
%matplotlib inline

In [12]:
p = (pmatrx+1).float()
# normalizing with 1 so that the probabilty for unseen sequence is not 0

In [13]:
p = p/p.sum(2, keepdim=True)
# calculating probabilities...

In [14]:
p[1, 1].sum()
# prob is 1 now

tensor(1.0000)

In [15]:
# sampling from the distribution..

for i in range(10):
    # we want 10 samples
    ix1 = 0    # init to first start token char.
    ix2 = 0    # init to second start token char. as we have 2 '.' token in the beginning...
    out = []    # output array to store chars
    while True:
        # we have '.' as a start/end padding character, and we start with '.' and stay in this loop of predicting next char untill ix = 0 i.e the end char token.
        pro = p[ix1, ix2] # first row with '.' '.', basically we're asking for row of probs for character that start with '.', '.' / are starting chars.
        ix3 = torch.multinomial(pro, num_samples=1, replacement=True).item() # replacement is true because we can have same letter as output next, they can repeat.
        ix1, ix2 = ix2, ix3
        if ix3 == 0:
            break
        out.append(itos[ix3])
    print("".join(out))

camalayla
brass
ivphish
brehmaxance
jayura
tad
sadelynn
bren
ruharlte
kenrtia


In [16]:
# negative log likelihood estimation
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1, ix2, ix3 = stoi[ch1], stoi[ch2], stoi[ch3]
        prob = p[ix1, ix2, ix3]
        log_prob = torch.log(prob)
        log_likelihood += log_prob
        n += 1

print("Log Likelihood: ", log_likelihood.item())
nll = -log_likelihood
print(f"Negative Log Likelihood: {nll}")
print(f"Normalized NLL: {nll/n}")

Log Likelihood:  -504653.0
Negative Log Likelihood: 504653.0
Normalized NLL: 2.2119739055633545


In [17]:
import torch.nn.functional as F

In [20]:
# creating training data for trigrams
xs, ys = [], [], []
for w in words:
    w = w.lower()
    chs = ['.'] + ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)
        zs.append(ix3)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
zs = torch.tensor(zs)
num = zs.nelement()
print("Num of examples: ", num)

# weight matrix
W = torch.randn((54,27), requires_grad=True)

Num of examples:  228146


In [28]:
# training loop
x1_enc = F.one_hot(xs, num_classes=27).float()
x2_enc = F.one_hot(ys, num_classes=27).float()
xenc = torch.cat([x1_enc, x2_enc], dim=1)

for k in range(10000):


    logits = xenc @ W
    counts = logits.exp()
    probs = counts/counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(num), zs].log().mean()
    print("Loss: ", loss.item())

    W.grad = None
    loss.backward()

    W.data += -5 * W.grad # here i am using higher learning rate because descent was way too slow, but at the end I decreased the learning rate...

print("Loss after training: ", loss.item())

Loss:  2.377272605895996
Loss:  2.360490322113037
Loss:  2.3533411026000977
Loss:  2.3499085903167725
Loss:  2.348121404647827
Loss:  2.347130298614502
Loss:  2.346548080444336
Loss:  2.3461859226226807
Loss:  2.345947027206421
Loss:  2.3457798957824707
Loss:  2.345656394958496
Loss:  2.3455607891082764
Loss:  2.3454837799072266
Loss:  2.345419406890869
Loss:  2.345364570617676
Loss:  2.3453173637390137
Loss:  2.34527587890625
Loss:  2.3452391624450684
Loss:  2.3452060222625732
Loss:  2.3451764583587646
Loss:  2.345149278640747
Loss:  2.345125198364258
Loss:  2.3451027870178223
Loss:  2.3450825214385986
Loss:  2.3450636863708496
Loss:  2.345046043395996
Loss:  2.3450300693511963
Loss:  2.3450145721435547
Loss:  2.3450005054473877
Loss:  2.344987154006958
Loss:  2.3449745178222656
Loss:  2.3449628353118896
Loss:  2.344951629638672
Loss:  2.3449409008026123
Loss:  2.344930410385132
Loss:  2.3449208736419678
Loss:  2.3449113368988037
Loss:  2.344902753829956
Loss:  2.34489369392395
Loss: 

KeyboardInterrupt: 

In [29]:
print("Final loss after training: ", loss.item())

Final loss after training:  2.339594841003418


In [30]:
# observation:
# statistical approach yielded better results, as the negative log likeliihood was around 2.21 vs 2.33 in neural network approach.

In [ ]:
# Generating words..